# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARIFULISLAM-7/Machine-Learning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [10]:
### 1. Unit of analysis + time window

# **One row = one aggregated statistical block group in California, as defined by the 1990 US Census.**

# There is no explicit time window within the dataset itself, as it represents a static snapshot from the 1990 US Census.

### 2. Fields: feature / label / context / excluded

Based on a typical machine learning task of predicting housing values, the fields are categorized as follows:

*   **Label:**
    *   `median_house_value`: The median house value for California districts.

*   **Features:**
    *   `longitude`: A measure of how far west a house is; a higher value is farther west.
    *   `latitude`: A measure of how far north a house is; a higher value is farther north.
    *   `housing_median_age`: Median age of a house within a block; a lower number is a newer house.
    *   `total_rooms`: Total number of rooms within a block.
    *   `total_bedrooms`: Total number of bedrooms within a block.
    *   `population`: Total number of people residing within a block.
    *   `households`: Total number of households, a group of people residing within a home unit, for a block.
    *   `median_income`: Median income for households within a block of houses (measured in tens of thousands of US Dollars).

*   **Context:** No specific context fields distinct from features.

*   **Excluded:** None of the available columns are excluded, as they all contribute to the housing value prediction.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify the number of rows, which represents the number of block groups.
print(f"Number of rows (block groups): {len(df_housing)}")

Number of rows (block groups): 17000


### 3. Verify it with queries (grain, counts, missing values, windows)

#### Missing Values
No missing values were found across any of the columns in the dataset, indicating good data completeness.

#### Row Counts & Duplicates (Grain)
The dataset contains 17,000 rows, each intended to represent an aggregated statistical block group.

*   **Exact Duplicate Rows:** No exact duplicate rows (where all column values are identical) were found.
*   **Duplicate (Longitude, Latitude) Pairs:** However, `5,946` rows were found to have duplicate `(longitude, latitude)` pairs. This suggests that the `(longitude, latitude)` coordinates do not uniquely identify each block group. It might imply that:
    1.  Multiple distinct block groups are assigned the same geographical centroid coordinates due to rounding.
    2.  The granularity of the `(longitude, latitude)` data is coarser than the block group, and several block groups fall within the same reported coordinate.

This finding slightly refines our understanding of the 'unit of analysis' and the uniqueness implied by geographical coordinates. While each row is a block group, its exact location might not be unique by `(longitude, latitude)` alone.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify column existence and data types
print("DataFrame columns and their data types:")
print(df_housing.info())

# Check for expected columns explicitly
expected_columns = [
    'longitude', 'latitude', 'housing_median_age', 'total_rooms',
    'total_bedrooms', 'population', 'households', 'median_income',
    'median_house_value'
]

missing_columns = [col for col in expected_columns if col not in df_housing.columns]
if missing_columns:
    print(f"\nError: The following expected columns are missing: {missing_columns}")
else:
    print("\nAll expected columns are present.")

DataFrame columns and their data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           17000 non-null  float64
 1   latitude            17000 non-null  float64
 2   housing_median_age  17000 non-null  float64
 3   total_rooms         17000 non-null  float64
 4   total_bedrooms      17000 non-null  float64
 5   population          17000 non-null  float64
 6   households          17000 non-null  float64
 7   median_income       17000 non-null  float64
 8   median_house_value  17000 non-null  float64
dtypes: float64(9)
memory usage: 1.2 MB
None

All expected columns are present.


### 4. Data limits

Based on the analysis, here are the data limitations:

*   **Geographical Precision:** The `longitude` and `latitude` fields are not unique identifiers for each block group, as evidenced by `5,946` duplicate pairs. This means the model should not rely on these coordinates for exact, unique spatial identification of every single block group, but rather as approximate location indicators. Decisions requiring fine-grained spatial distinctions based solely on these coordinates might be inaccurate.

*   **Temporal Scope:** The data represents a static snapshot from the 1990 US Census. It cannot be used to infer trends or make predictions about housing values or demographics *after* 1990. Any insights derived are specific to that period.

*   **Feature Completeness:** The dataset includes basic demographic and housing structure information. It lacks other potentially important factors influencing housing value, such as:
    *   Local amenities (schools, parks, shopping).
    *   Economic indicators beyond median income (e.g., employment rates, industry presence).
    *   Property-specific details (e.g., age of individual homes, number of bathrooms, lot size, specific architectural styles, recent renovations).
    *   Environmental factors (e.g., proximity to natural hazards, air quality).

*   **Bias from Aggregation:** The data is aggregated at the block group level. While useful, this means individual household or property-level variations are smoothed out. Conclusions drawn from this data might not apply directly to individual properties but rather to statistical averages within a block group.

*   **Income Measure:** `median_income` is in tens of thousands of US dollars. While useful, it is a single metric that may not fully capture the economic diversity or spending power within a block group.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Check for missing values
print("\n--- Missing Values Check ---")
missing_values = df_housing.isnull().sum()
print("Number of missing values per column:")
print(missing_values.to_markdown())

if missing_values.sum() == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing values found in the dataset. Please investigate.")

# 2. Check for duplicate rows (grain verification using key identifiers)
print("\n--- Duplicate Rows Check ---")
# For housing data, longitude and latitude often uniquely identify a location.
# We can combine them or assume that a combination of several key features should be unique for a block group.
# Let's check duplicates based on all columns for now, then refine if needed.
duplicates = df_housing.duplicated().sum()
if duplicates == 0:
    print("No exact duplicate rows found in the dataset.")
else:
    print(f"Found {duplicates} duplicate rows. Please investigate.")

# For a more specific 'grain' check, we could check for duplicates on 'longitude' and 'latitude' combination
duplicates_lat_lon = df_housing.duplicated(subset=['longitude', 'latitude']).sum()
if duplicates_lat_lon == 0:
    print("No duplicate (longitude, latitude) pairs found.")
else:
    print(f"Found {duplicates_lat_lon} duplicate (longitude, latitude) pairs. This might indicate issues with the 'one row = one block group' assumption if these pairs are expected to be unique.")


--- Missing Values Check ---
Number of missing values per column:
|                    |   0 |
|:-------------------|----:|
| longitude          |   0 |
| latitude           |   0 |
| housing_median_age |   0 |
| total_rooms        |   0 |
| total_bedrooms     |   0 |
| population         |   0 |
| households         |   0 |
| median_income      |   0 |
| median_house_value |   0 |
No missing values found in the dataset.

--- Duplicate Rows Check ---
No exact duplicate rows found in the dataset.
Found 5946 duplicate (longitude, latitude) pairs. This might indicate issues with the 'one row = one block group' assumption if these pairs are expected to be unique.


In [3]:
import pandas as pd

file_path = 'sample_data/california_housing_train.csv'

if os.path.exists(file_path):
    print(f"Loading dataset: {file_path}")
    df_housing = pd.read_csv(file_path)
    print("Dataset loaded successfully. Displaying first 5 rows:")
    print(df_housing.head().to_markdown(index=False))
    print("\nDataset info:")
    df_housing.info()
else:
    print(f"Error: The file '{file_path}' was not found. Cannot proceed with data contract definition without the dataset.")

Loading dataset: sample_data/california_housing_train.csv
Dataset loaded successfully. Displaying first 5 rows:
|   longitude |   latitude |   housing_median_age |   total_rooms |   total_bedrooms |   population |   households |   median_income |   median_house_value |
|------------:|-----------:|---------------------:|--------------:|-----------------:|-------------:|-------------:|----------------:|---------------------:|
|     -114.31 |      34.19 |                   15 |          5612 |             1283 |         1015 |          472 |          1.4936 |                66900 |
|     -114.47 |      34.4  |                   19 |          7650 |             1901 |         1129 |          463 |          1.82   |                80100 |
|     -114.56 |      33.69 |                   17 |           720 |              174 |          333 |          117 |          1.6509 |                85700 |
|     -114.57 |      33.64 |                   14 |          1501 |              337 |          51

In [1]:
import os

# Check if the file exists before trying to read it
file_path = 'skills/README.md'
if os.path.exists(file_path):
    !cat {file_path}
else:
    print(f"Error: The file '{file_path}' was not found.")
    print("Please make sure the 'skills' directory and 'README.md' file exist in the correct location.")

Error: The file 'skills/README.md' was not found.
Please make sure the 'skills' directory and 'README.md' file exist in the correct location.


In [2]:
import os

# Check if 'sample_data/README.md' exists
sample_readme_path = 'sample_data/README.md'
if os.path.exists(sample_readme_path):
    print(f"Reading contents of {sample_readme_path}:")
    !cat {sample_readme_path}
else:
    print(f"Error: The file '{sample_readme_path}' was not found either.")
    print("Please provide the correct path to the README file or the details of the 'skill' directly.")

Reading contents of sample_data/README.md:
This directory includes a few sample datasets to get you started.

*   `california_housing_data*.csv` is California housing data from the 1990 US
    Census; more information is available at:
    https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub

*   `mnist_*.csv` is a small sample of the
    [MNIST database](https://en.wikipedia.org/wiki/MNIST_database), which is
    described at: http://yann.lecun.com/exdb/mnist/

*   `anscombe.json` contains a copy of
    [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet); it
    was originally described in

    Anscombe, F. J. (1973). 'Graphs in Statistical Analysis'. American
    Statistician. 27 (1): 17-21. JSTOR 2682899.

    and our copy was prepared by the
    [vega_datasets library](https://github.com/altair-viz/vega_datasets/blob/4f67bdaad10f45e3549984e17e1b3088c731503d/vega_datasets/_data/anscombe.js

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.